200. 岛屿数量

https://leetcode.cn/problems/number-of-islands/description/?envType=study-plan-v2&envId=top-interview-150

给你一个由 '1'（陆地）和 '0'（水）组成的的二维网格，请你计算网格中岛屿的数量。
岛屿总是被水包围，并且每座岛屿只能由水平方向和、或竖直方向上相邻的陆地连接形成。
此外，你可以假设该网格的四条边均被水包围。

如下写了一个加四周水边框的脚本，但之后就没有思路了；我隐隐知道要用树，将图连通问题转化为基于树结构的遍历问题，但是比较生疏就先问了 AI。AI 的回答是 DFS 或者 BFS，或者并查集，这个我们稍后再说。

In [2]:
from typing import List

class Solution:
    def numIslands(self, grid: List[List[str]]) -> int:
        for i, g in enumerate(grid):
            grid[i] = ["0"] + g + ["0"]
        grid_aug = [['0'] * len(grid[0])]
        for g in grid:
            grid_aug.append(g)
        grid_aug.append(['0'] * len(grid[0]))
        print(grid_aug)

solution = Solution()
grid = [
  ['1','1','1','1','0'],
  ['1','1','0','1','0'],
  ['1','1','0','0','0'],
  ['0','0','0','0','0']
]
solution.numIslands(grid)

[['0', '0', '0', '0', '0', '0', '0'], ['0', '1', '1', '1', '1', '0', '0'], ['0', '1', '1', '0', '1', '0', '0'], ['0', '1', '1', '0', '0', '0', '0'], ['0', '0', '0', '0', '0', '0', '0'], ['0', '0', '0', '0', '0', '0', '0']]


首先是 DFS，也是最第一性的（相比于 BFS）。这里最巧妙的实现就在于，当我们涂色的时候，不需要再额外创造一个颜色（目前已经有陆地和水），而是直接将陆地**变成水**即可，因为这丝毫不影响后续岛屿的判定。这是一种很优雅的做法。其次，需要注意的另一个细节在于 dfs 函数实现时的越界管理。最后，别忘了是 `'1'` 而不是 `1`。

In [5]:
class Solution:
    def numIslands(self, grid: List[List[str]]) -> int: 
        ans = 0
        rows, cols = len(grid), len(grid[0])
        def dfs(i, j):
            if i < 0 or i > rows - 1 or j < 0 or j > cols - 1 or grid[i][j] == '0':
                return
            grid[i][j] = '0'
            dfs(i - 1, j)
            dfs(i + 1, j)
            dfs(i, j - 1)
            dfs(i, j + 1)
        for i in range(rows):
            for j in range(cols):
                if grid[i][j] == '1':
                    ans += 1
                    dfs(i, j)
        return ans
    
grid = [["1","1","0","0","0"],["1","1","0","0","0"],["0","0","1","0","0"],["0","0","0","1","1"]]
solution = Solution()
solution.numIslands(grid)

3

之后的方法后续更新，因为实际上没有本质上的创新。

130. 被围绕的区域

https://leetcode.cn/problems/surrounded-regions/description/?envType=study-plan-v2&envId=top-interview-150

给你一个 m x n 的矩阵 board ，由若干字符 'X' 和 'O' 组成，捕获所有被围绕的区域：

- 连接：一个单元格与水平或垂直方向上相邻的单元格连接。
- 区域：连接所有 'O' 的单元格来形成一个区域。
- 围绕：如果一个区域中的所有 'O' 单元格都不在棋盘的边缘，则该区域被包围。这样的区域 完全 被 'X' 单元格包围。
- 通过 原地 将输入矩阵中的所有 'O' 替换为 'X' 来 捕获被围绕的区域。你不需要返回任何值。

让我们先看看我们自己的自大做法，以及其背后的问题。本质上来说，这题因为图的经验不足，我们的大脑先入为主地希望将上一题（岛屿数量）的 DFS 思路直接套用进来，因为这在直觉上确实很相似，即找寻被围的部分；但是，本质区别有一个，就是如果本题的“岛屿”在地图边缘，那么它就不算一个岛屿了！这看上去微不足道的边界条件变化，实际上决定了一个很 critical 的先后问题：我们在理论上就不可能像上一题一样，在**递归的过程**中修改 'O' 为 'X'，而是只可能先标记，后修改。如下我们看为什么。

In [20]:
class Solution:
    def solve(self, board: List[List[str]]) -> None:
        """
        Do not return anything, modify board in-place instead.
        """
        rows = len(board)
        cols = len(board[0])
        checked = [[0] * cols] * rows

        def dfs(i, j):
            if board[i][j] == "X":
                return True
            elif j == 0 or j == (cols - 1) or i == 0 or i == (rows - 1):
                return False
            elif checked[i][j] == 1:
                checked[i][j] = 0
                return True
            else:
                checked[i][j] = 1

            if dfs(i + 1, j) and dfs(i - 1, j) and dfs(i, j + 1) and dfs(i, j - 1):
                board[i][j] = "X"
                return True
            else:
                return False

        for i in range(1, rows - 1):
            for j in range(1, cols - 1):
                if board[i][j] == "O":
                    dfs(i, j)
                    checked = [[0] * cols] * rows

solution = Solution()
board = [["X","X","X","X"],["X","O","O","X"],["X","X","O","X"],["X","O","X","X"]]
solution.solve(board)
board

[['X', 'X', 'X', 'X'],
 ['X', 'X', 'X', 'X'],
 ['X', 'X', 'X', 'X'],
 ['X', 'O', 'X', 'X']]

如上是我自己第一遍写的代码，在测试用例上通过了，就以为万事大吉了。在正式讨论为什么不能在递归回溯时原地翻转之前，我们先 debug 一个小问题。

In [21]:
checked = [[0] * 3] * 5
checked[2][2] = 1
checked

[[0, 0, 1], [0, 0, 1], [0, 0, 1], [0, 0, 1], [0, 0, 1]]

`checked = [[0] * cols] * rows` 的写法是错误的！`* rows` 会让每一行都指向同一个列表对象，因此正确的写法如下。

在改掉这一部分后，我们运行了测试发现仍旧不通过的 case 如下。可以看到，`board[2][3]` 的位置被错误地翻转了，而它本应该是不被翻转的。

In [18]:
from typing import List

class Solution:
    def solve(self, board: List[List[str]]) -> None:
        """
        Do not return anything, modify board in-place instead.
        """
        rows = len(board)
        cols = len(board[0])
        checked = [[0] * cols for _ in range(rows)]

        def dfs(i, j):
            if board[i][j] == "X":
                return True
            elif j == 0 or j == (cols - 1) or i == 0 or i == (rows - 1):
                return False
            elif checked[i][j] == 1:
                checked[i][j] = 0
                return True
            else:
                checked[i][j] = 1

            if dfs(i + 1, j) and dfs(i - 1, j) and dfs(i, j + 1) and dfs(i, j - 1):
                board[i][j] = "X"
                return True
            else:
                return False

        for i in range(1, rows - 1):
            for j in range(1, cols - 1):
                if board[i][j] == "O":
                    dfs(i, j)
                    if board[2][3] == "X":
                        print(i, j)
                    checked = [[0] * cols for _ in range(rows)]

solution = Solution()
board = [["O","X","X","O","X"],["X","O","O","X","O"],["X","O","X","O","X"],["O","X","O","O","O"],["X","X","O","X","O"]]
solution.solve(board)
board

3 3


[['O', 'X', 'X', 'O', 'X'],
 ['X', 'X', 'X', 'X', 'O'],
 ['X', 'X', 'X', 'X', 'X'],
 ['O', 'X', 'O', 'O', 'O'],
 ['X', 'X', 'O', 'X', 'O']]

在经过探针分析后我们看到，是 `board[3][3]` 作为入口时，出现的误判；原因也很显而易见，由于 `board[3][3]` 的位置被标为了 `checked[i][j] = 1`，导致该位置被判定为 all true，因此直接翻转了；这明显是错误的，`checked[i][j] = 1` 不仅浪费了空间，还没有起到其应有的、传达信息的作用；我们在最开始加入这个矩阵的目的就是在于防止陷入死循环，然而这种暴力的做法还是使我们身陷囹圄。

是否可以沿着这个方向改进？没必要。如果继续依托 checked 矩阵改进，就会出现一个悖论：我们希望不陷入死循环，但又希望当前位置 $s$ 能够在当前时刻就直接翻转；可是，位置 $s$ 的翻转与否是依赖于 $s-1$ 的；又已知 位置 $s-1$ 的翻转是依赖于 $s$ 的，这也是 DFS 算法被选的理由，因此二者的判定是相互依赖的，而基于深度为方向的单向 DFS 无法做到这一点。

因此，放弃以上想法后，本题的核心在于：从边界出发，标记所有不会被捕获的 O；先把所有与边界连通的 O 标记成安全，最后把剩下的 O 全部翻成 X。这里，我们也接触到了图算法的第二种处理模式，即先标记、后解决，而不是在递归过程中就立刻解决；其次就是，先标记的情况一定是 BFS 或者说一定是从边界条件出发的，因为边界条件出发才能保证 $s-1$ 一定是确定的，而 $s$ 就可以依赖之继续被标记。

In [30]:
from collections import deque
q = deque()
q.append(1)
q.append(2)
q.popleft()
q

deque([2])

在正式开始前，我们也需要再次熟悉 python 中队列的写法。队列通常和 BFS 广度优先搜索是绑定的，可以直接依托 deque 实现。队列的核心思想是先入先出 FIFO，因此在使用上最常用的两个方法即 `append` 和 `popleft`，如上。另一方面，通常使用 `while q` 语句来完成队列的执行。

In [31]:
from collections import deque

class Solution:
    def solve(self, board: List[List[str]]) -> None:
        """
        Do not return anything, modify board in-place instead.
        """
        rows = len(board)
        cols = len(board[0])
        q = deque()

        def add(r, c):
            if 0 <= r < rows and 0 <= c < cols and board[r][c] == "O":
                board[r][c] = "#"
                q.append((r, c))
        
        for r in range(rows):
            add(r, 0)
            add(r, cols - 1)
        for c in range(cols):
            add(0, c)
            add(rows - 1, c)

        while q:
            r, c = q.popleft()
            add(r + 1, c)
            add(r - 1, c)
            add(r, c + 1)
            add(r, c - 1)
        
        for r in range(rows):
            for c in range(cols):
                if board[r][c] == "O":
                    board[r][c] = "X"
                elif board[r][c] == "#":
                    board[r][c] = "O"

最后，占位符是一件很优雅的事，可以经常使用 “#” 来占位。如上的做法相当于从边界的 O 出发，进行广度优先搜索，实际上就是入队列。注意 `q.append((r, c))` 的位置。